# day-30-deploy-rag-api — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [8]:
# ---- Solution 1 ----
ALLOWED = {"question", "top_k"}
def validate_body(body):
    errs = []
    extra = set(body) - ALLOWED
    if extra: errs.append(f"unexpected keys: {sorted(extra)}")
    if not isinstance(body.get("question"), str): errs.append("question must be a string")
    tk = body.get("top_k", 3)
    if not (isinstance(tk, int) and 1 <= tk <= 10): errs.append("top_k must be int in 1..10")
    return errs

for b in [{"question": "hi"}, {"question": 5}, {"question": "x", "top_k": 99},
          {"question": "x", "junk": 1}, {}]:
    e = validate_body(b)
    print(f"{str(b)[:40]:42s} -> {'OK' if not e else e}")

{'question': 'hi'}                         -> OK
{'question': 5}                            -> ['question must be a string']
{'question': 'x', 'top_k': 99}             -> ['top_k must be int in 1..10']
{'question': 'x', 'junk': 1}               -> ["unexpected keys: ['junk']"]
{}                                         -> ['question must be a string']


In [9]:
# ---- Solution 2 ----
_IDEM = {}   # module-level; real: DynamoDB with TTL
def handler_idempotent(event, context=None):
    key = (event.get("headers") or {}).get("idempotency-key")
    if key and key in _IDEM:
        r = dict(_IDEM[key]); r["headers"] = {**r["headers"], "x-idempotent-replay": "true"}
        return r
    r = handler(event, context)
    if key and r["statusCode"] == 200:
        _IDEM[key] = r
    return r

ev = api_event(body={"question": "how long do refunds take"}, headers={"idempotency-key": "k1"})
r1 = handler_idempotent(ev); r2 = handler_idempotent(ev)
print("first :", r1["statusCode"], r1["headers"].get("x-idempotent-replay"))
print("replay:", r2["statusCode"], r2["headers"].get("x-idempotent-replay"), "| same body:", r1["body"] == r2["body"])

{"evt": "rag_request", "trace_id": "ad07318f77944858", "q_len": 24, "n_sources": 1, "latency_ms": 0.0, "cold_start_ms": 153.9}
first : 200 None
replay: 200 true | same body: True


In [10]:
# ---- Solution 3 ----
def handler_timeout_aware(event, context, downstream_ms=5000):
    remaining = context.get_remaining_time_in_millis()
    if remaining < downstream_ms + 500:
        return _response(503, {"error": "insufficient time budget, retry"}, "t-guard") | \
               {"headers": {"retry-after": "2", "content-type": "application/json"}}
    return handler(event, context)

class TightCtx(Ctx):
    def get_remaining_time_in_millis(self): return 3000   # only 3s left
print("S3:", handler_timeout_aware(api_event(body={"question": "x"}), TightCtx())["statusCode"],
      "(503 -- bail early with Retry-After instead of being killed mid-call)")

S3: 503 (503 -- bail early with Retry-After instead of being killed mid-call)


In [11]:
# ---- Solution 6 ----
GB_SEC_PRICE = 0.0000133334 * (0.8)     # arm64 ~20% cheaper
REQ_PRICE = 0.20 / 1e6
APIGW_PRICE = 1.00 / 1e6
def lambda_month(rps, dur_s, mem_mb):
    reqs = rps * 3600 * 730
    return reqs*REQ_PRICE + reqs*APIGW_PRICE + reqs*dur_s*(mem_mb/1024)*GB_SEC_PRICE
FARGATE_TASK_MONTH = (0.04048*0.5 + 0.004445*1) * 730   # 0.5 vCPU + 1GB, on-demand
for rps in [1, 10, 30, 100, 300]:
    lm = lambda_month(rps, 0.4, 1024)
    n_tasks = max(1, -(-int(rps*0.4)//1))   # ~rps*dur concurrent -> tasks (1 req each, rough)
    fg = n_tasks * FARGATE_TASK_MONTH
    print(f"{rps:>4} rps: lambda ${lm:>10,.0f}   fargate(~{n_tasks} task) ${fg:>9,.0f}   "
          f"{'lambda' if lm < fg else 'fargate'}")

   1 rps: lambda $        14   fargate(~1 task) $       18   lambda
  10 rps: lambda $       144   fargate(~4 task) $       72   fargate
  30 rps: lambda $       431   fargate(~12 task) $      216   fargate
 100 rps: lambda $     1,437   fargate(~40 task) $      721   fargate
 300 rps: lambda $     4,310   fargate(~120 task) $    2,162   fargate


### Solutions 4 & 5 (sketch)

**S4:** more memory → proportionally more vCPU → lower `duration`, but `GB-seconds` cost is
`mem × duration`. Cost is often flat-to-U-shaped (faster execution offsets the higher memory
rate up to a point); latency keeps dropping. Pick the memory at the bottom of the cost curve
unless latency forces higher. AWS Lambda Power Tuning runs this as a Step Function against your
real function.

**S5:** `DeploymentPreference: { Type: Canary10Percent5Minutes, Alarms: [!Ref ErrorAlarm] }`;
`ErrorAlarm` on the function's `Errors` metric (or a custom `5xx` metric from your logs),
threshold e.g. `> 1%` of invocations over 2 datapoints of 1 minute. CodeDeploy shifts 10% of
traffic, watches the alarm for 5 minutes, then shifts the rest — or auto-rolls-back on alarm.

### Answer key
1. Module scope runs once per container (cold start) and is reused across all warm
   invocations; putting it in `handler` re-does the expensive setup on every request.
2. 29 seconds (hard). Options: stream the response via a Lambda **Function URL** (up to 15
   min), move to Fargate/App Runner behind an ALB, or make the RAG call faster (smaller model,
   less context, tighter `max_tokens`).
3. Keep **no model in the Lambda** — embeddings and generation via API (Bedrock / Knowledge
   Base), retrieval via a query to the vector DB — so the function is thin and cold-starts in
   ~200 ms.
4. API Gateway enforces the key and rate limits *before* invoking Lambda, so a throttled or
   unauthorized caller costs you nothing (no Lambda invocation, no compute).
5. It caps how many copies of the function run concurrently — bounding your cost blast radius
   and protecting downstream resources (Bedrock token/req quotas, the DB connection pool) from
   a traffic spike.
6. Any two: requests > 29s → Fargate/App Runner or Function-URL streaming; steady high RPS →
   an always-on container (Fargate/App Runner/EKS); needs a GPU → ECS-GPU/EKS/SageMaker/
   Bedrock; WebSockets → API Gateway WebSocket or Fargate.
7. The `trace_id` (returned in every response as `x-trace-id` and in the 500 body). Then run a
   CloudWatch Logs Insights query on that id to see the request's structured log line and, via
   X-Ray, the downstream calls — localise the failing stage (Day 27) and add the input as an
   eval case (Day 26).